### Lectura de ruta

In [1]:
import os
import json
import pandas as pd
from sqlalchemy import create_engine

# -----------------------------
# RUTA BASE RAW
# -----------------------------
BASE_PATH = r"C:\Users\catia\Desktop\prestashop_data\raw\source=prestashop"

print("Ruta RAW:", BASE_PATH)
print("Existe la ruta:", os.path.exists(BASE_PATH))

Ruta RAW: C:\Users\catia\Desktop\prestashop_data\raw\source=prestashop
Existe la ruta: True


### Función para leer endpoint

In [4]:
def read_endpoint(endpoint_name):
    endpoint_path = os.path.join(BASE_PATH, f"asset={endpoint_name}")
    
    # buscar último ingest_date
    ingest_dates = sorted(os.listdir(endpoint_path))
    latest_ingest = ingest_dates[-1]
    
    ingest_path = os.path.join(endpoint_path, latest_ingest)
    
    # buscar último run_id
    run_ids = sorted(os.listdir(ingest_path))
    latest_run = run_ids[-1]
    
    run_path = os.path.join(ingest_path, latest_run)
    
    file_path = os.path.join(run_path, f"{endpoint_name}.json")
    
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    df = pd.DataFrame(data["records"])
    
    print(f"Endpoint leído: {endpoint_name}")
    print("Filas:", len(df))
    
    return df

#### Prueba de lectura

In [7]:
df_categories = read_endpoint("categories")
df_categories.head()

Endpoint leído: categories
Filas: 10


,category_id,category_name
0,1,Categoria 1
1,2,Categoria 2
2,3,Categoria 3
3,4,Categoria 4
4,5,Categoria 5


In [6]:
df_products = read_endpoint("products")
df_products.shape

Endpoint leído: products
Filas: 80


(80, 4)

In [8]:
df_customers = read_endpoint("customers")
df_customers.shape

Endpoint leído: customers
Filas: 4000


(4000, 4)

In [9]:
df_orders = read_endpoint("orders")
df_orders.shape

Endpoint leído: orders
Filas: 61607


(61607, 9)

## Conexión a SQL

In [11]:
from sqlalchemy import text

server = '.\\SQLEXPRESS'
database = 'prestashop_dw'

connection_string = (
    f"mssql+pyodbc://@{server}/{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)

# prueba conexión
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Conexión OK")

C:\Users\catia\AppData\Local\Temp\ipykernel_12660\1885976182.py:15: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Conexión OK


### Vaciar tablas (full load)

In [13]:
with engine.connect() as conn:
    conn.execute(text("DELETE FROM dbo.fact_sales"))
    conn.execute(text("DELETE FROM dbo.dim_product"))
    conn.execute(text("DELETE FROM dbo.dim_customer"))
    conn.execute(text("DELETE FROM dbo.dim_category"))
    conn.execute(text("DELETE FROM dbo.dim_channel"))
    conn.execute(text("DELETE FROM dbo.dim_date"))
    conn.commit()

print("Tablas vaciadas correctamente")

Tablas vaciadas correctamente


### Construcción y carga dim_date

In [ ]:
# Leer calendarios
df_marketing = read_endpoint("marketing_calendar")
df_promotions = read_endpoint("promotions_calendar")

print("Marketing:", df_marketing.shape)
print("Promotions:", df_promotions.shape)

Endpoint leído: marketing_calendar
Filas: 730
Endpoint leído: promotions_calendar
Filas: 730
Marketing: (730, 2)
Promotions: (730, 2)


In [15]:
# Convertir fecha a datetime
df_marketing["date"] = pd.to_datetime(df_marketing["date"])
df_promotions["date"] = pd.to_datetime(df_promotions["date"])

# Merge por fecha
df_date = df_marketing.merge(
    df_promotions,
    on="date",
    how="left"
)

# Generar columnas derivadas
df_date["date_key"] = df_date["date"].dt.strftime("%Y%m%d").astype(int)
df_date["year"] = df_date["date"].dt.year
df_date["month"] = df_date["date"].dt.month
df_date["month_name"] = df_date["date"].dt.month_name()
df_date["day"] = df_date["date"].dt.day
df_date["weekday"] = df_date["date"].dt.weekday
df_date["is_weekend"] = df_date["weekday"].apply(lambda x: 1 if x >= 5 else 0)

# Renombrar columnas según modelo SQL
df_date = df_date.rename(columns={
    "date": "full_date",
    "campaña_marketing_activa": "is_marketing_active",
    "hay_promocion": "has_promotion"
})

# Seleccionar columnas en orden final
df_date = df_date[[
    "date_key",
    "full_date",
    "year",
    "month",
    "month_name",
    "day",
    "weekday",
    "is_weekend",
    "is_marketing_active",
    "has_promotion"
]]

df_date.head()

,date_key,full_date,year,month,month_name,day,weekday,is_weekend,is_marketing_active,has_promotion
0,20240208,2024-02-08,2024,2,February,8,3,0,0,0
1,20240209,2024-02-09,2024,2,February,9,4,0,0,0
2,20240210,2024-02-10,2024,2,February,10,5,1,0,0
3,20240211,2024-02-11,2024,2,February,11,6,1,1,0
4,20240212,2024-02-12,2024,2,February,12,0,0,0,0


In [ ]:
# Carga de dim_date 
df_date.to_sql(
    "dim_date",
    engine,
    schema="dbo",
    if_exists="append",
    index=False
)

print("dim_date cargada correctamente")

### Carga de dim_category

In [18]:
df_categories = read_endpoint("categories")
df_categories.head()

Endpoint leído: categories
Filas: 10


,category_id,category_name
0,1,Categoria 1
1,2,Categoria 2
2,3,Categoria 3
3,4,Categoria 4
4,5,Categoria 5


In [19]:
df_categories.to_sql(
    "dim_category",
    engine,
    schema="dbo",
    if_exists="append",
    index=False
)

print("dim_category cargada correctamente")

dim_category cargada correctamente


### Carga de dim_channel 

In [20]:
# Leer endpoint
df_channels = read_endpoint("channels")
df_channels

Endpoint leído: channels
Filas: 2


,channel_id,channel_name
0,1,web
1,2,mobile


In [21]:
df_channels.to_sql(
    "dim_channel",
    engine,
    schema="dbo",
    if_exists="append",
    index=False
)

print("dim_channel cargada correctamente")

dim_channel cargada correctamente


### Carga de dim_customer

In [ ]:
# Leer datos
df_customers = read_endpoint("customers")
df_customers.head()

Endpoint leído: customers
Filas: 4000


,customer_id,first_purchase_date,customer_type,country
0,1,2024-05-21,new,ES
1,2,2025-03-12,new,ES
2,3,2024-08-23,new,PT
3,4,2024-12-27,new,ES
4,5,2024-09-28,new,FR


In [24]:
# Transformar fecha
df_customers["first_purchase_date"] = pd.to_datetime(
    df_customers["first_purchase_date"]
)

In [25]:
# Cargar datos
df_customers.to_sql(
    "dim_customer",
    engine,
    schema="dbo",
    if_exists="append",
    index=False
)
print("dim_customer cargada correctamente")

dim_customer cargada correctamente


### Cargar dim_product

In [ ]:
# Leer productos
df_products = read_endpoint("products")
df_products.head()

Endpoint leído: products
Filas: 80


,product_id,product_name,category_id,base_price
0,1,Producto 1,4,196.11
1,2,Producto 2,4,250.90
2,3,Producto 3,10,252.05
3,4,Producto 4,2,10.59
4,5,Producto 5,2,18.55


In [28]:
# obtener category_key
df_dim_category = pd.read_sql(
    "SELECT category_key, category_id FROM dbo.dim_category",
    engine
)
df_dim_category.head()

,category_key,category_id
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


In [29]:
# Merge para obtener category_key
df_products = df_products.merge(
    df_dim_category,
    on="category_id",
    how="left"
)
df_products.head()

,product_id,product_name,category_id,base_price,category_key
0,1,Producto 1,4,196.11,4
1,2,Producto 2,4,250.90,4
2,3,Producto 3,10,252.05,10
3,4,Producto 4,2,10.59,2
4,5,Producto 5,2,18.55,2


In [30]:
# Seleccionar columnas finales según modelo SQL
df_products_final = df_products[[
    "product_id",
    "product_name",
    "base_price",
    "category_key"
]]
df_products_final.head()

,product_id,product_name,base_price,category_key
0,1,Producto 1,196.11,4
1,2,Producto 2,250.90,4
2,3,Producto 3,252.05,10
3,4,Producto 4,10.59,2
4,5,Producto 5,18.55,2


In [31]:
# Insertar
df_products_final.to_sql(
    "dim_product",
    engine,
    schema="dbo",
    if_exists="append",
    index=False
)
print("dim_product cargada correctamente")

dim_product cargada correctamente


### Construcción y carga fact_sales

In [32]:
# Leer orders
df_orders = read_endpoint("orders")
df_orders.head()

Endpoint leído: orders
Filas: 61607


,order_id,order_date,customer_id,channel_id,order_status,total_tax_excl,is_new_customer,main_category_id,main_product_id
0,1,2024-02-08,1346,1,cancelled,170.29,0,6,15
1,2,2024-02-08,954,1,paid,283.39,0,10,3
2,3,2024-02-08,152,2,paid,231.24,0,4,2
3,4,2024-02-08,3690,1,paid,202.73,0,4,1
4,5,2024-02-08,875,1,paid,180.11,0,4,1


In [33]:
# Preparar fecha y generar date_key
# Convertir order_date a datetime
df_orders["order_date"] = pd.to_datetime(df_orders["order_date"])

# Generar date_key
df_orders["date_key"] = df_orders["order_date"].dt.strftime("%Y%m%d").astype(int)
df_orders[["order_date", "date_key"]].head()

,order_date,date_key
0,2024-02-08,20240208
1,2024-02-08,20240208
2,2024-02-08,20240208
3,2024-02-08,20240208
4,2024-02-08,20240208


In [34]:
# Obtener las dimensiones desde SQL
df_dim_date = pd.read_sql(
    "SELECT date_key FROM dbo.dim_date",
    engine)

df_dim_customer = pd.read_sql(
    "SELECT customer_key, customer_id FROM dbo.dim_customer",
    engine)

df_dim_product = pd.read_sql(
    "SELECT product_key, product_id, category_key FROM dbo.dim_product",
    engine)

df_dim_category = pd.read_sql(
    "SELECT category_key, category_id FROM dbo.dim_category",
    engine)

df_dim_channel = pd.read_sql(
    "SELECT channel_key, channel_id FROM dbo.dim_channel",
    engine)

print("Dimensiones cargadas desde SQL")

Dimensiones cargadas desde SQL


In [36]:
# Merge con dim_customer
df_orders = df_orders.merge(
    df_dim_customer,
    on="customer_id",
    how="left"
)

df_orders.head()

,order_id,order_date,customer_id,channel_id,order_status,total_tax_excl,is_new_customer,main_category_id,main_product_id,date_key,customer_key
0,1,2024-02-08,1346,1,cancelled,170.29,0,6,15,20240208,1346
1,2,2024-02-08,954,1,paid,283.39,0,10,3,20240208,954
2,3,2024-02-08,152,2,paid,231.24,0,4,2,20240208,152
3,4,2024-02-08,3690,1,paid,202.73,0,4,1,20240208,3690
4,5,2024-02-08,875,1,paid,180.11,0,4,1,20240208,875


In [37]:
# Merge con dim_product
df_orders = df_orders.merge(
    df_dim_product[["product_key", "product_id"]],
    left_on="main_product_id",
    right_on="product_id",
    how="left"
)
df_orders.head()

,order_id,order_date,customer_id,channel_id,order_status,total_tax_excl,is_new_customer,main_category_id,main_product_id,date_key,customer_key,product_key,product_id
0,1,2024-02-08,1346,1,cancelled,170.29,0,6,15,20240208,1346,15,15
1,2,2024-02-08,954,1,paid,283.39,0,10,3,20240208,954,3,3
2,3,2024-02-08,152,2,paid,231.24,0,4,2,20240208,152,2,2
3,4,2024-02-08,3690,1,paid,202.73,0,4,1,20240208,3690,1,1
4,5,2024-02-08,875,1,paid,180.11,0,4,1,20240208,875,1,1


In [39]:
# Comprobación nulos
df_orders["product_key"].isnull().sum()

np.int64(0)

In [40]:
# Merge con dim_category
df_orders = df_orders.merge(
    df_dim_category,
    left_on="main_category_id",
    right_on="category_id",
    how="left"
)
df_orders.head()

,order_id,order_date,customer_id,channel_id,order_status,total_tax_excl,is_new_customer,main_category_id,main_product_id,date_key,customer_key,product_key,product_id,category_key,category_id
0,1,2024-02-08,1346,1,cancelled,170.29,0,6,15,20240208,1346,15,15,6,6
1,2,2024-02-08,954,1,paid,283.39,0,10,3,20240208,954,3,3,10,10
2,3,2024-02-08,152,2,paid,231.24,0,4,2,20240208,152,2,2,4,4
3,4,2024-02-08,3690,1,paid,202.73,0,4,1,20240208,3690,1,1,4,4
4,5,2024-02-08,875,1,paid,180.11,0,4,1,20240208,875,1,1,4,4


In [41]:
# Comprobación nulos
df_orders["category_key"].isnull().sum()

np.int64(0)

In [42]:
# Merge con dim_channel
df_orders = df_orders.merge(
    df_dim_channel,
    on="channel_id",
    how="left"
)
df_orders.head()

,order_id,order_date,customer_id,channel_id,order_status,total_tax_excl,is_new_customer,main_category_id,main_product_id,date_key,customer_key,product_key,product_id,category_key,category_id,channel_key
0,1,2024-02-08,1346,1,cancelled,170.29,0,6,15,20240208,1346,15,15,6,6,1
1,2,2024-02-08,954,1,paid,283.39,0,10,3,20240208,954,3,3,10,10,1
2,3,2024-02-08,152,2,paid,231.24,0,4,2,20240208,152,2,2,4,4,2
3,4,2024-02-08,3690,1,paid,202.73,0,4,1,20240208,3690,1,1,4,4,1
4,5,2024-02-08,875,1,paid,180.11,0,4,1,20240208,875,1,1,4,4,1


In [43]:
df_orders["channel_key"].isnull().sum()

np.int64(0)

In [44]:
# Construcción de fact_sales
df_fact = df_orders[[
    "order_id",
    "date_key",
    "customer_key",
    "product_key",
    "category_key",
    "channel_key",
    "order_status",
    "total_tax_excl",
    "is_new_customer"
]].copy()
df_fact.head()

,order_id,date_key,customer_key,product_key,category_key,channel_key,order_status,total_tax_excl,is_new_customer
0,1,20240208,1346,15,6,1,cancelled,170.29,0
1,2,20240208,954,3,10,1,paid,283.39,0
2,3,20240208,152,2,4,2,paid,231.24,0
3,4,20240208,3690,1,4,1,paid,202.73,0
4,5,20240208,875,1,4,1,paid,180.11,0


In [45]:
df_fact.isnull().sum()

order_id           0
date_key           0
customer_key       0
product_key        0
category_key       0
channel_key        0
order_status       0
total_tax_excl     0
is_new_customer    0
dtype: int64

In [46]:
# Insertar datos
df_fact.to_sql(
    "fact_sales",
    engine,
    schema="dbo",
    if_exists="append",
    index=False
)
print("fact_sales cargada correctamente")

fact_sales cargada correctamente


In [47]:
# Validar coherencia de totales
df_fact["total_tax_excl"].sum()

np.float64(10165978.09)